## 섹션 0-1: 환경 감지 및 경로 설정

1. `git clone https://github.com/ehdgus4173/DACON_Structure_Stability.git`
2. 레포 상위 폴더에 `dataset/` 폴더 생성 후 데이콘 데이터 배치:
   ```
   DACON_Structure_Stability/  ← 레포
   dataset/                    ← 데이터 (레포와 같은 위치)
     train/ dev/ test/
     train.csv  dev.csv  sample_submission.csv
   ```
3. conda 환경 생성: `conda create -n stability python=3.10`
4. 패키지 설치: `pip install -r requirements.txt`
5. Jupyter 커널 등록: `python -m ipykernel install --user --name stability`
6. 피처 추출: `python src/features/extract_base.py && python src/features/extract_advanced.py`
7. 이 노트북 실행

In [ ]:
import os, sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_DIR    = '/kaggle/input/datasets/junseopkim/structure-stability-data/data'
    SRC_DIR     = '/kaggle/input/datasets/junseopkim/structure-stability-src/src'
    OUT_DIR     = '/kaggle/working/outputs'
    FEAT_DIR    = '/kaggle/working/features'
    IMG_SIZE    = 384
    NUM_WORKERS = 2
else:
    REPO_ROOT = Path('..').resolve()
    sys.path.insert(0, str(REPO_ROOT))
    from config import DATASET_DIR, PROJECT_ROOT, PHYS_COLS_V2
    DATA_DIR    = str(DATASET_DIR)
    SRC_DIR     = str(REPO_ROOT / 'src')
    OUT_DIR     = str(REPO_ROOT / 'outputs')
    FEAT_DIR    = str(REPO_ROOT / 'features')
    IMG_SIZE    = 224
    NUM_WORKERS = 0

sys.path.append(SRC_DIR)
FEATURES_CSV = os.path.join(FEAT_DIR, 'combined_features_v3.csv')

FIG_DIR = f"{OUT_DIR}/../reports/figures" if not IS_KAGGLE else f"{OUT_DIR}/figures"
os.makedirs(f'{OUT_DIR}/models',      exist_ok=True)
os.makedirs(f'{OUT_DIR}/logs',        exist_ok=True)
os.makedirs(f'{OUT_DIR}/submissions', exist_ok=True)
os.makedirs(FEAT_DIR,                 exist_ok=True)
os.makedirs(FIG_DIR,                  exist_ok=True)

print(f"환경: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"FEATURES_CSV: {FEATURES_CSV}")
print(f"IMG_SIZE: {IMG_SIZE}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")

## 섹션 0-1b: 물리 피처 추출 (최초 1회만 실행)
combined_features_v3.csv가 없을 때만 extract_base.py → extract_advanced.py 순서로 실행합니다.

In [ ]:
import subprocess

if not os.path.exists(FEATURES_CSV):
    print('Feature file not found - extracting (5~10 min)...')
    env = os.environ.copy()
    if IS_KAGGLE:
        env['KAGGLE_MODE']  = '1'
        env['DATASET_DIR']  = DATA_DIR
        env['FEAT_OUT_DIR'] = FEAT_DIR
    r1 = subprocess.run(
        [sys.executable, os.path.join(SRC_DIR, 'features', 'extract_base.py')],
        capture_output=True, text=True, env=env
    )
    print(r1.stdout[-800:] if r1.stdout else '')
    if r1.returncode != 0:
        print('extract_base error:', r1.stderr[-500:])
    r2 = subprocess.run(
        [sys.executable, os.path.join(SRC_DIR, 'features', 'extract_advanced.py')],
        capture_output=True, text=True, env=env
    )
    print(r2.stdout[-800:] if r2.stdout else '')
    if r2.returncode != 0:
        print('extract_advanced error:', r2.stderr[-500:])
    print(f'Done: {FEATURES_CSV}')
else:
    print(f'Already exists: {FEATURES_CSV}')

import pandas as pd
feat_df = pd.read_csv(FEATURES_CSV)
print(f'Feature shape: {feat_df.shape}')


## 섹션 0-2: Import 및 DEVICE 설정


In [ ]:
import importlib, sys

# 세션 재시작 없이도 최신 버전 강제 로드
for mod_name in ['experiment_utils', 'dataset', 'model', 'augmentation']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from model import MultiViewNet
from experiment_utils import set_seed, EarlyStopping, run_experiment, run_inference, train_one_epoch, evaluate

import random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 섹션 1-2: 모델 구조 확인


In [ ]:
test_model = MultiViewNet(backbone_name='resnet18', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out = test_model(dummy, dummy)
print(f"출력 shape: {out.shape}")
print(f"총 파라미터: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model, dummy

## 섹션 1-3: 베이스 Config 사전 정의
dh 브랜치 기준 — PHYS_COLS_V2 21개 (top-view 2세대 피처)
실행 순서: 0-1 → 0-1b → 0-2 → 1-2 → 1-3 → 원하는 EXP

In [ ]:
from config import PHYS_COLS_V2
PHYS_COLS_DH = PHYS_COLS_V2
TOP10_COLS = ['f_cx_offset', 'compact_ecc', 't_ecc_2d', 't_compactness', 'top_grid_tilt_angle', 'mass_asymmetry_2d', 'front_grid_tilt_angle', 'height_support_risk', 't_left_mass_ratio', 'front_grid_perspective_ratio']
TOP13_COLS = TOP10_COLS + ["f_cy_ratio","t_cy_offset","support_margin_min"]

config_exp008 = {
    "exp_id": "008", "model_version": "8", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "diff_concat",
    "use_physics": True, "physics_dim": 6,
    "features_csv": FEATURES_CSV,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "custom", "dev_split_ratio": 0.8, "img_size": IMG_SIZE,
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 7, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
config_exp012 = {
    **config_exp008,
    "exp_id": "012", "model_version": "12",
    "dev_split_ratio": 0.5, "early_stopping_patience": 10
}
print(f"PHYS_COLS_DH: {len(PHYS_COLS_DH)}개, TOP10: {len(TOP10_COLS)}개, TOP13: {len(TOP13_COLS)}개")
print("config_exp008 / config_exp012 정의 완료")


## 섹션 2: EXP-001 — HIGH 증강 (Ablation)


In [ ]:
config_exp001 = {
    "exp_id": "001", "model_version": "2", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "concat",
    "use_physics": False, "physics_dim": 6,
    "features_csv": FEATURES_CSV,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "imagenet", "dev_split_ratio": 0.0, "img_size": IMG_SIZE,
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 5, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp001 = run_experiment(config_exp001)

## 섹션 3: EXP-002 — Dev 학습 포함 (Ablation)


In [ ]:
config_exp002 = {**config_exp001, "exp_id": "002", "model_version": "3", "dev_split_ratio": 0.8}
result_exp002 = run_experiment(config_exp002)

## 섹션 4: EXP-004 — Custom 정규화 (Ablation)


In [ ]:
config_exp004 = {**config_exp001, "exp_id": "004", "model_version": "4", "norm_version": "custom"}
result_exp004 = run_experiment(config_exp004)

## 섹션 5: EXP-005 — EfficientNet-B0 백본 (Ablation)


In [ ]:
config_exp005 = {**config_exp001, "exp_id": "005", "model_version": "5",
                 "model_name": "efficientnet", "backbone": "efficientnet_b0"}
result_exp005 = run_experiment(config_exp005)

## 섹션 6: EXP-006 — 물리 피처 6개 (Ablation)


In [ ]:
config_exp006 = {**config_exp001, "exp_id": "006", "model_version": "6",
                 "use_physics": True, "physics_dim": 6}
result_exp006 = run_experiment(config_exp006)

## 섹션 7: EXP-007 — diff_concat Fusion (Ablation)


In [ ]:
config_exp007 = {**config_exp001, "exp_id": "007", "model_version": "7", "fusion_mode": "diff_concat"}
result_exp007 = run_experiment(config_exp007)

## 섹션 8: EXP-008 — 최적 조합 ✅ Public 0.16137 / 175위
Custom 정규화 + diff_concat Fusion + Dev 포함 + 물리 피처 6개 + HIGH 증강
결과: Best Val LogLoss 0.1028 (Epoch 2) — Public 0.16137 / 175위

In [ ]:
config_exp008 = {
    "exp_id": "008", "model_version": "8", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "diff_concat",
    "use_physics": True, "physics_dim": 6,
    "features_csv": FEATURES_CSV,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "custom", "dev_split_ratio": 0.8, "img_size": IMG_SIZE,
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 7, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp008 = run_experiment(config_exp008)

## 섹션 9: EXP-009~011 — dev_split_ratio Ablation (0.0 / 0.5 / 1.0)


In [ ]:
for ratio, exp_id, ver in [(0.0, '009', '9'), (0.5, '010', '10'), (1.0, '011', '11')]:
    cfg = {
        **config_exp008,
        "exp_id": exp_id, "model_version": ver,
        "dev_split_ratio": ratio,
        "epochs": 3 if ratio == 1.0 else 50,
        "early_stopping_patience": 7
    }
    result = run_experiment(cfg)
    print(f"ratio={ratio} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

## 섹션 10: EXP-012 — 최적 조합 + ratio=0.5 ✅ Val 0.0608
결과: Best Val LogLoss 0.0608 (Epoch 19)

In [ ]:
config_exp012 = {
    **config_exp008,
    "exp_id": "012", "model_version": "12",
    "dev_split_ratio": 0.5,
    "early_stopping_patience": 10
}
result_exp012 = run_experiment(config_exp012)

## 섹션 11: 추론 — submission_v2.csv (EXP-008 기준)
Public LogLoss: 0.16137 / 175위


In [ ]:
preds_v2 = run_inference(config=config_exp008, model_path=result_exp008['model_path'])
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = preds_v2
sub_df['stable_prob']   = 1.0 - preds_v2
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v2.csv", index=False)
print(f"저장 완료 | prob 범위: {preds_v2.min():.4f}~{preds_v2.max():.4f}")

## 섹션 12: EXP-013 — CosineAnnealingLR
결과: Best Val LogLoss 0.0756 (Epoch 13)

In [ ]:
config_exp013 = {
    **config_exp012,
    "exp_id": "013", "model_version": "13",
    "lr_scheduler": "cosine", "lr_min": 1e-5,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp013 = run_experiment(config_exp013)

## 섹션 13: EXP-014 — EfficientNet-B4 백본
결과: Best Val LogLoss 0.3281 (Epoch 1)

In [ ]:
config_exp014 = {
    **config_exp012,
    "exp_id": "014", "model_version": "14",
    "model_name": "efficientnet_b4", "backbone": "efficientnet_b4",
    "batch_size": 8, "lr_scheduler": "cosine", "lr_min": 1e-5,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp014 = run_experiment(config_exp014)

## 섹션 14: EXP-015 — Seed 앙상블 (42/43/44) ✅ Val 0.0438
결과: seed44=0.0438 (전체 최고) — Public 0.07936 / 100위

In [ ]:
seed_preds = []
for seed in [42, 43, 44]:
    cfg = {
        **config_exp012,
        "exp_id": f"015_seed{seed}", "model_version": f"15s{seed}",
        "random_state": seed, "epochs": 50, "early_stopping_patience": 10
    }
    result = run_experiment(cfg)
    preds  = run_inference(config=cfg, model_path=result['model_path'])
    seed_preds.append(preds)
    print(f"seed={seed} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

ensemble_preds = np.mean(seed_preds, axis=0)
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = ensemble_preds
sub_df['stable_prob']   = 1.0 - ensemble_preds
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v3_ensemble.csv", index=False)
print(f"앙상블 완료 | prob 범위: {ensemble_preds.min():.4f}~{ensemble_preds.max():.4f}")

## 섹션 15: EXP-016 — Phys MLP 구조
결과: Best Val LogLoss 0.1328 (Epoch 5)

In [ ]:
config_exp016 = {
    **config_exp012,
    "exp_id": "016", "model_version": "16",
    "use_phys_mlp": True, "epochs": 50, "early_stopping_patience": 10
}
result_exp016 = run_experiment(config_exp016)

## 섹션 16: 추론 — submission_v3.csv (EXP-012 기준)


In [ ]:
preds_v3 = run_inference(config=config_exp012, model_path=result_exp012['model_path'])
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = preds_v3
sub_df['stable_prob']   = 1.0 - preds_v3
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v3.csv", index=False)
print(f"저장 완료 | prob 범위: {preds_v3.min():.4f}~{preds_v3.max():.4f}")

## [구버전] 섹션 17: EXP-017 — 팀원 피처 20개 (physics_dim 6→20)
physics_dim 6 → 20 (팀원 전체 피처)

In [ ]:
config_exp017 = {
    **config_exp012,
    "exp_id": "017", "model_version": "17",
    "use_physics": True, "physics_dim": 20,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp017 = run_experiment(config_exp017)

## [구버전] 섹션 18: EXP-018 — 팀원 피처 20개 + Phys MLP


In [ ]:
config_exp018 = {
    **config_exp017,
    "exp_id": "018", "model_version": "18",
    "use_phys_mlp": True,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp018 = run_experiment(config_exp018)

## [구버전] 섹션 19: EXP-019 — 격자 피처 Ablation (physics_dim=14)


In [ ]:
config_exp019 = {
    **config_exp012,
    "exp_id": "019", "model_version": "19",
    "use_physics": True, "physics_dim": 14,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp019 = run_experiment(config_exp019)

## [구버전] 섹션 20: EXP-020 — Seed 앙상블 (팀원 피처 최적 config) → submission_v5.csv
EXP-017 vs EXP-018 결과 보고 더 좋은 config로 변경 후 실행


In [ ]:
# EXP-017 vs EXP-018 결과 보고 더 좋은 것으로 변경
best_phys_config = config_exp017

seed_preds_v5 = []
for seed in [42, 43, 44]:
    cfg = {
        **best_phys_config,
        "exp_id": f"020_seed{seed}", "model_version": f"20s{seed}",
        "random_state": seed, "epochs": 50, "early_stopping_patience": 10
    }
    result = run_experiment(cfg)
    preds  = run_inference(config=cfg, model_path=result['model_path'])
    seed_preds_v5.append(preds)
    print(f"seed={seed} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

ensemble_v5 = np.mean(seed_preds_v5, axis=0)
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = np.clip(ensemble_v5, 1e-7, 1-1e-7)
sub_df['stable_prob']   = 1.0 - sub_df['unstable_prob']
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v5.csv", index=False)
print(f"submission_v5.csv 저장 완료 | prob 범위: {ensemble_v5.min():.4f}~{ensemble_v5.max():.4f}")

## 섹션 30: EXP-030 — dh 기준 TOP10 (SHAP gain>2.5, dim=10)
f_cx_offset(1위), compact_ecc(2위), t_ecc_2d(3위) 등 dh 2세대 핵심 포함

In [ ]:
config_exp030 = {
    **config_exp012,
    "exp_id": "030", "model_version": "30",
    "use_physics": True, "physics_dim": len(TOP10_COLS),
    "phys_cols": TOP10_COLS,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp030 = run_experiment(config_exp030)
print(f"EXP-030 TOP10 Val LogLoss: {result_exp030['best_val_logloss']:.4f}")


## 섹션 31: EXP-031 — dh 기준 TOP13 (SHAP gain>2.0, dim=13)
TOP10 + f_cy_ratio, t_cy_offset, support_margin_min

In [ ]:
config_exp031 = {
    **config_exp012,
    "exp_id": "031", "model_version": "31",
    "use_physics": True, "physics_dim": len(TOP13_COLS),
    "phys_cols": TOP13_COLS,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp031 = run_experiment(config_exp031)
print(f"EXP-031 TOP13 Val LogLoss: {result_exp031['best_val_logloss']:.4f}")


## 섹션 32: EXP-032 — dh 유효 18개 (gain>0, dim=18)
gain=0 피처(t_compactness_sq, grid_detected x2) 제거

In [ ]:
VALID_18 = [c for c in PHYS_COLS_DH
            if c not in ["t_compactness_sq","front_grid_detected","top_grid_detected"]]
print(f"유효 피처: {len(VALID_18)}개")
config_exp032 = {
    **config_exp012,
    "exp_id": "032", "model_version": "32",
    "use_physics": True, "physics_dim": len(VALID_18),
    "phys_cols": VALID_18,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp032 = run_experiment(config_exp032)
print(f"EXP-032 유효18 Val LogLoss: {result_exp032['best_val_logloss']:.4f}")


## 섹션 33: EXP-033 — 최적 config seed 앙상블 → submission_v8.csv
EXP-030/031/032 결과 보고 best_dh_config 수정 후 실행

In [ ]:
best_dh_config = config_exp030  # 결과 보고 수정
seed_preds_v8 = []
for seed in [42, 43, 44]:
    cfg = {
        **best_dh_config,
        "exp_id": f"033_s{seed}", "model_version": f"33s{seed}",
        "random_state": seed
    }
    res   = run_experiment(cfg)
    preds = run_inference(config=cfg, model_path=res["model_path"])
    seed_preds_v8.append(preds)
    print(f"seed={seed} Val: {res['best_val_logloss']:.4f}")

from scipy.special import logit, expit
ens   = np.mean(seed_preds_v8, axis=0)
final = expit(logit(np.clip(ens, 1e-7, 1-1e-7)) / 1.0)
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub["unstable_prob"] = np.clip(final, 1e-7, 1-1e-7)
sub["stable_prob"]   = 1.0 - sub["unstable_prob"]
sub.to_csv(f"{OUT_DIR}/submissions/submission_v8.csv", index=False)
print(f"submission_v8.csv 저장 완료")
print(f"범위: {ens.min():.4f}~{ens.max():.4f}")


## 섹션 34: EXP-034 — ViT-B/16 백본
CNN과 다른 inductive bias → 앙상블 다양성 확보
lr=5e-5, batch=4 (T4 VRAM 대응)

In [ ]:
config_exp034 = {
    **config_exp012,
    "exp_id": "034", "model_version": "34",
    "model_name": "vit_b_16", "backbone": "vit_b_16",
    "batch_size": 4, "lr": 5e-5,
    "lr_scheduler": "cosine", "lr_min": 1e-6,
    "epochs": 30, "early_stopping_patience": 10
}
result_exp034 = run_experiment(config_exp034)
print(f"EXP-034 ViT Val LogLoss: {result_exp034['best_val_logloss']:.4f}")


## 섹션 35: EXP-035 — Optuna 하이퍼파라미터 탐색 (n_trials=20)
탐색: lr, dev_split_ratio, early_stopping_patience

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    cfg = {
        **config_exp012,
        "exp_id": f"opt{trial.number}",
        "model_version": f"opt{trial.number}",
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "dev_split_ratio": trial.suggest_float("dev_split_ratio", 0.3, 0.7),
        "early_stopping_patience": trial.suggest_int("patience", 5, 15),
        "random_state": 42
    }
    return run_experiment(cfg)["best_val_logloss"]

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, timeout=3600)
print(f"Best Val: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


## 섹션 36: EXP-036 — XGBoost + CNN 가중 앙상블
dh 피처 기반 XGBoost, Val 기반 가중치 최적화

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import log_loss as sk_logloss

feat_xgb = pd.read_csv(FEATURES_CSV)
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
dev_df   = pd.read_csv(f"{DATA_DIR}/dev.csv")
all_df   = pd.concat([train_df, dev_df], ignore_index=True)
all_df["id"]    = all_df["id"].astype(str)
feat_xgb["id"] = feat_xgb["id"].astype(str)
num_cols = [c for c in PHYS_COLS_DH if c in feat_xgb.columns]
merged   = all_df.merge(feat_xgb[["id"]+num_cols], on="id", how="inner")
X = merged[num_cols].fillna(0).values.astype("float32")
y = (merged["label"]=="unstable").astype(int).values

xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, eval_metric="logloss", random_state=42)
xgb.fit(X, y)

test_ids  = sorted([d for d in os.listdir(f"{DATA_DIR}/test")
                    if os.path.isdir(f"{DATA_DIR}/test/{d}")])
test_feat = feat_xgb[feat_xgb["id"].isin(test_ids)].sort_values("id")
xgb_preds = xgb.predict_proba(test_feat[num_cols].fillna(0).values)[:,1]
print(f"XGBoost 추론 완료")
print(f"범위: {xgb_preds.min():.4f}~{xgb_preds.max():.4f}")


## 섹션 37: EXP-037 — TTA (Test-Time Augmentation)
재학습 없이 추론 시 증강 적용 — 원본+hflip+vflip+rot90+rot180 평균

In [ ]:
from advanced_utils import run_inference_tta
# EXP-033 최고 seed 모델에 TTA 적용
best_model_path = f"{OUT_DIR}/models/resnet18_v33s44_best.pth"
cfg_tta = {
    **best_dh_config,
    "exp_id": "033_s44", "model_version": "33s44", "random_state": 44
}
preds_tta = run_inference_tta(config=cfg_tta, model_path=best_model_path, n_augments=5)
print(f"TTA 완료")
print(f"범위: {preds_tta.min():.4f}~{preds_tta.max():.4f}")


## 섹션 38: 최종 앙상블 + Temperature Scaling → 최종 제출
Val 기반 T 최적화 후 submission_final.csv 생성

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.special import logit, expit
from sklearn.metrics import log_loss as sk_logloss
import torch
from dataset import MultiViewDataset
from model import MultiViewNet
from torchvision import transforms

# 사용할 모델 목록 — 실험 결과 보고 수정
ENSEMBLE_MODELS = [
    # (config, model_path) 형태로 추가
    # (cfg_033_s42, f"{OUT_DIR}/models/resnet18_v33s42_best.pth"),
]

mean, std = [0.4611,0.4359,0.3905],[0.2193,0.2150,0.2109]
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])
dev_csv_df = pd.read_csv(f"{DATA_DIR}/dev.csv")
feat_df    = pd.read_csv(FEATURES_CSV)

test_preds_all, val_preds_all, val_labels_arr = [], [], []

for cfg, model_path in ENSEMBLE_MODELS:
    test_preds_all.append(run_inference(config=cfg, model_path=model_path))
    phys_cols = cfg.get("phys_cols", None)
    val_ds = MultiViewDataset(
        df=dev_csv_df, root_dir=f"{DATA_DIR}/dev",
        transform=val_tf, feature_df=feat_df, feature_cols=phys_cols
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS
    )
    model = MultiViewNet(
        backbone_name=cfg["backbone"], fusion_mode=cfg["fusion_mode"],
        use_physics=cfg["use_physics"], physics_dim=cfg.get("physics_dim",6),
        img_size=IMG_SIZE
    ).to(DEVICE)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for batch in val_loader:
            views, feats, labels = batch
            probs = torch.sigmoid(
                model(views[0].to(DEVICE), views[1].to(DEVICE), feats.to(DEVICE))
            ).cpu().numpy().flatten()
            vp.extend(np.clip(probs, 1e-7, 1-1e-7))
            vl.extend(labels.numpy())
    val_preds_all.append(np.array(vp))
    if len(val_labels_arr) == 0:
        val_labels_arr = np.array(vl)

if test_preds_all:
    val_avg  = np.mean(val_preds_all,  axis=0)
    test_avg = np.mean(test_preds_all, axis=0)
    res = minimize_scalar(
        lambda T: sk_logloss(
            val_labels_arr,
            expit(logit(np.clip(val_avg, 1e-7, 1-1e-7)) / T)
        ),
        bounds=(0.1, 5.0), method="bounded"
    )
    T_opt  = res.x
    before = sk_logloss(val_labels_arr, val_avg)
    after  = sk_logloss(
        val_labels_arr,
        expit(logit(np.clip(val_avg,1e-7,1-1e-7))/T_opt)
    )
    print(f"최적 T: {T_opt:.4f}")
    print(f"Val before: {before:.4f} | after: {after:.4f}")
    final = expit(logit(np.clip(test_avg, 1e-7, 1-1e-7)) / T_opt)
    sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
    sub["unstable_prob"] = np.clip(final, 1e-7, 1-1e-7)
    sub["stable_prob"]   = 1.0 - sub["unstable_prob"]
    sub.to_csv(f"{OUT_DIR}/submissions/submission_final.csv", index=False)
    print(f"submission_final.csv 저장 완료")
    print(f"범위: {final.min():.4f}~{final.max():.4f}")
else:
    print("ENSEMBLE_MODELS가 비어있음 — 모델 경로를 추가하세요")


## 섹션 21: 전체 실험 결과 비교표


In [ ]:
# ================================================================
# 섹션 21: 전체 실험 결과 비교표 (자동 수집)
# ================================================================

# 1. 하드코딩된 기존 결과 (EXP-001~016, v4 제출)
results_fixed = [
    {"exp_id": "baseline", "변경사항": "없음(기준점)",         "val": 1.5369, "public": 1.67225, "비고": ""},
    {"exp_id": "001",      "변경사항": "HIGH 증강",             "val": 0.5007, "public": None,    "비고": ""},
    {"exp_id": "002",      "변경사항": "Dev 포함(0.8)",         "val": 0.3687, "public": None,    "비고": ""},
    {"exp_id": "004",      "변경사항": "Custom 정규화",          "val": 0.1700, "public": None,    "비고": "단독 최대 효과"},
    {"exp_id": "005",      "변경사항": "EfficientNet-B0",       "val": 0.5081, "public": None,    "비고": ""},
    {"exp_id": "006",      "변경사항": "물리 피처 6개",          "val": 0.3438, "public": None,    "비고": ""},
    {"exp_id": "007",      "변경사항": "diff_concat Fusion",    "val": 0.1996, "public": None,    "비고": ""},
    {"exp_id": "008",      "변경사항": "최적 조합",              "val": 0.1028, "public": 0.16137, "비고": "1차 제출"},
    {"exp_id": "009",      "변경사항": "ratio=0.0",             "val": 0.1880, "public": None,    "비고": ""},
    {"exp_id": "010",      "변경사항": "ratio=0.5",             "val": 0.1037, "public": None,    "비고": ""},
    {"exp_id": "011",      "변경사항": "ratio=1.0",             "val": 0.5896, "public": None,    "비고": ""},
    {"exp_id": "012",      "변경사항": "최적조합+ratio=0.5",    "val": 0.0608, "public": None,    "비고": "베이스 config 확정"},
    {"exp_id": "013",      "변경사항": "CosineAnnealingLR",     "val": 0.0756, "public": None,    "비고": ""},
    {"exp_id": "014",      "변경사항": "EfficientNet-B4",       "val": 0.3281, "public": None,    "비고": "VRAM 부족"},
    {"exp_id": "015_s42",  "변경사항": "Seed42",               "val": 0.0994, "public": None,    "비고": ""},
    {"exp_id": "015_s43",  "변경사항": "Seed43",               "val": 0.0491, "public": None,    "비고": ""},
    {"exp_id": "015_s44",  "변경사항": "Seed44",               "val": 0.0438, "public": None,    "비고": "단일 모델 최고"},
    {"exp_id": "016",      "변경사항": "Phys MLP",              "val": 0.1328, "public": None,    "비고": ""},
    {"exp_id": "v4",       "변경사항": "Seed 앙상블+TempScale", "val": None,   "public": 0.07936, "비고": "현재 최고 제출 100위"},
]

# 2. 현재 세션 실험 결과 자동 수집
session_results = []
result_vars = {
    "017": ("팀원 피처 20개",             "result_exp017"),
    "018": ("팀원 피처 20개 + PhysMLP",   "result_exp018"),
    "019": ("격자 제외 dim=14",           "result_exp019"),
    "019a":("gain=0 제거 dim=16",         "result_exp019a"),
    "019b":("C+D 제외 dim=9",            "result_exp019b"),
    "019c":("A만 dim=5",                 "result_exp019c"),
    "021": ("Label Smoothing 0.05",       "result_exp021"),
    "022": ("SWA",                        "result_exp022"),
    "023": ("Pseudo-Label",               "result_exp023"),
    "025": ("ViT-B/16",                   "result_exp025"),
}

for exp_id, (desc, var_name) in result_vars.items():
    try:
        res = eval(var_name)
        session_results.append({
            "exp_id": exp_id,
            "변경사항": desc,
            "val": round(res['best_val_logloss'], 4),
            "public": None,
            "비고": ""
        })
    except NameError:
        pass  # 미실행 실험은 스킵

# EXP-020 seed 앙상블
try:
    session_results.append({
        "exp_id": "020",
        "변경사항": "팀원 피처 seed 앙상블",
        "val": None,
        "public": None,
        "비고": "submission_v5.csv 생성"
    })
except:
    pass

# XGBoost
try:
    _ = xgb_preds
    session_results.append({
        "exp_id": "026",
        "변경사항": "XGBoost + CNN 앙상블",
        "val": None,
        "public": None,
        "비고": "submission_v6 생성"
    })
except NameError:
    pass

# 3. 합치기 및 출력
all_results = results_fixed + session_results
result_df = pd.DataFrame(all_results)
result_df = result_df.sort_values('val', na_position='last').reset_index(drop=True)

print("=" * 75)
print(f"{'순위':>4} {'exp_id':>10} {'변경사항':30} {'Val':>8} {'Public':>8} {'비고'}")
print("=" * 75)
rank = 1
for _, row in result_df.iterrows():
    val_str    = f"{row['val']:.4f}" if pd.notna(row['val']) else "  -   "
    pub_str    = f"{row['public']:.5f}" if pd.notna(row['public']) else "  -    "
    note       = row['비고'] if pd.notna(row['비고']) else ""
    print(f"{rank:>4}. {row['exp_id']:>9} {row['변경사항']:30} {val_str:>8} {pub_str:>9}  {note}")
    rank += 1
print("=" * 75)
print(f"총 실험 수: {len(result_df)}개 | 현재 최고 Public: 0.07936 (100위)")

result_df.to_csv(f"{OUT_DIR}/logs/experiment_summary.csv", index=False)
print(f"저장: {OUT_DIR}/logs/experiment_summary.csv")
